In [1]:
import pandas as pd
import requests
import json

from datetime import datetime
import glob
from enum import Enum
from pathlib import Path

from aind_data_schema.core.procedures import SpecimenProcedure, ImmunolabelClass, HCRSeries, Antibody, Procedures, ViralMaterial, TarsVirusIdentifiers

from aind_data_schema_models.specimen_procedure_types import SpecimenProcedureType

from aind_data_schema_models.organizations import Organization

from aind_data_schema_models.pid_names import PIDName

from aind_data_schema_models.registry import Registry

import logging



materials_sheet = pd.read_excel("./Mouse Tracker - Molecular Anatomy.xlsx", sheet_name="Mouse Tracker - Molecular Anato", header=[0], converters={})


In [2]:
log_file_name = "./logging/log_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".log"
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
# create file handler which logs even debug messages
fh = logging.FileHandler(log_file_name, "w", "utf-8")
fh.setLevel(logging.DEBUG)

# create formatter and add it to the handlers
formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
fh.setFormatter(formatter)
# add the handlers to logger
logger.addHandler(fh)

In [3]:
subj_procedures = {}

download_files = False

files = glob.glob("./original_spec_files/*.json")


tars_models = {}

if download_files:
    for file in files:    
        print(file)
        subj_id = file.split("\\")[-1].split("_")[0].split(".")[0]
        print(subj_id)
        if int(subj_id) not in materials_sheet["Mouse ID"].tolist():
            print("not found")
            continue

        subj_row = materials_sheet.loc[materials_sheet["Mouse ID"] == int(subj_id)]
        
        if len(subj_id) != 6:
            continue

        for virus_number in [1,2,3]:
            item = None
            tars_id = subj_row[f"Virus{virus_number} ID"].values[0]
            logging.info(f"Processing {tars_id} for {subj_id}")
            print(tars_id)

            if pd.isna(tars_id):
                continue

            if tars_id not in tars_models.keys():
                request = requests.get(f"http://aind-metadata-service/tars_injection_materials/{tars_id}")
            else:
                continue

            if request.status_code == 404 or request.status_code == 500:
                print(f"{tars_id} model not found")
                continue

            print(f"code: {request.status_code}")

        
            item = request.json()

            if not item or not item['data']:
                continue

            if item['message'] == 'Valid Model.':
                logging.info(f"Valid model for {tars_id}: {item['data']}")
                tars_models[tars_id] = item['data']
            else:
                print(f"invalid model for {tars_id}")
                print(item['message'])

print(tars_models)

for key, value in tars_models.items():
    logging.info(f"Writing {key} for {value}")
    with open(f'./tars_info/{key}.json', 'w') as outfile:
        json.dump(value, outfile)

{}


In [4]:

tars_files = glob.glob("./tars_info/*.json")
tars_models = {}
for file in tars_files:
    logging.info(f"Reading file {file}")
    with open(file) as json_file:
        data = json.load(json_file)
        tars_models[str(data['tars_identifiers']["prep_lot_number"])] = data


In [5]:
sanity_checks = []

def get_inj_material(subj_id, virus_number):
    print(materials_sheet["Mouse ID"].tolist())
    print(type(subj_id))
    print(type(materials_sheet["Mouse ID"].tolist()[0]))
    if int(subj_id) not in materials_sheet["Mouse ID"].tolist():
        logging.info(f"Subject {subj_id} not found in materials sheet")
        return []
    
    materials = []

    subj_row = materials_sheet.loc[materials_sheet["Mouse ID"] == int(subj_id)]
    logging.info(f"subj_row: {subj_row}")

    virus = subj_row[f"Virus{virus_number}"].values[0]
    logging.info(f"virus{virus_number}: {virus}")
    if pd.isna(virus):
        logging.info(f"virus{virus_number} is NA")
        return None

    virus_id = subj_row[f"Virus{virus_number} ID" ].values[0] # use this to look up TARS info

    titer = subj_row[f"Virus{virus_number} Titer (GC/mL)"].values[0]
    dose = float(subj_row[f"Virus{virus_number} Dose (GC/mouse)"].values[0])
    volume = subj_row[f"Virus{virus_number} Volume Injected"].values[0]
    mix_volume = subj_row[f"Virus Mix Volume injected"].values[0]
    if pd.isna(mix_volume):
        mix_volume = .1
    else:
        if isinstance(mix_volume, str):
            mix_volume = float(mix_volume.split("u")[0])
        else:
            mix_volume = float(mix_volume)
        mix_volume = mix_volume*.0001
    logging.info(f"titer: {titer}")
    logging.info(f"mix vol: {mix_volume}")

    if not pd.isna(mix_volume):
        logging.info("FOUND MIX VOL: " + str(mix_volume))
    else:
        return None


    try:
        logging.info(f"titer: {titer}, dose: {dose}, volume: {volume}, mix volume: {mix_volume}")
        
        mix_titer = None
        if not pd.isna(mix_volume):
            
            logging.info("mix volume is not NA")
            mix_titer = int(dose/(mix_volume))
            logging.info(f"mix titer: {mix_titer}")
        
        if mix_titer is not None:
            logging.info("using mixed titer")
            titer = mix_titer
        else:
            logging.info("using original titer")

    except:
        logging.info("something missing")


    if pd.isna(titer): # actually, do the calculation for everything
        logging.info("titer is NA")
        
        
        if pd.isna(volume):
            logging.error(f"Volume is NA for material {virus_number} : {subj_id}, {virus}")
            return None
            

        if isinstance(volume, str):
            volume = float(volume.split("u")[0])
        logging.info(f"dose: {dose}, volume: {volume}")

        titer = int(dose/(volume*.001))
    else:
        try:
            titer = float(titer)
        except:
            logging.error(f"titer cannot be converted to float: {titer}")
            return None

    # do some checks to see how accurate titer is to dose/volume
        
    tars = None
    logging.info(f"virus_id: {virus_id}")
    logging.info(f"tars_models: {tars_models.keys()}")
    if str(virus_id) in tars_models.keys():
        logging.info(f"found tars for {virus_id}")
        tars = TarsVirusIdentifiers.model_validate(tars_models[virus_id]["tars_identifiers"])
        logging.info(f"tars for {subj_id}: {tars}")

    logging.info(f"titer: {titer}, dose: {dose}, volume: {volume}")
    new_material = ViralMaterial(
        name=virus,
        titer=titer,
        tars_identifiers=tars, 
    )

    logging.info(f"new material: {new_material}")

    logging.info(f"finished virus {virus_number}")

    return new_material


In [6]:
from decimal import Decimal
from math import isclose

def find_which_injection(subj_id, coordinates):
    subj_row = materials_sheet.loc[materials_sheet["Mouse ID"] == int(subj_id)]

    for val in range(1,4):
        coord = {}
        coord['ap'] = subj_row[f"Virus{val} AP (mm)"].values[0]
        coord['ml'] = subj_row[f"Virus{val} ML (mm)"].values[0]
        coord['dv'] = subj_row[f"Virus{val} DV (mm)"].values[0]

        if None in coord.values() or '?' in coord.values() or 'n/a' in coord.values():
            logging.error(f"Missing coordinates for {subj_id} {val}")
            continue

        logging.info(f"coord: {coord}")
        flag = True
        for key, value in coordinates.items():
            logging.info(f"coord {coord[key]} vs {value}")
            if 'L' in str(coord[key]) or 'R' in str(coord[key]) or '+' in str(coord[key]):
                continue
            if not isclose(coord[key], value):
                logging.info(f"coord {coord[key]} does not match {value}")
                logging.info(f"{type(coord[key])} vs {type(value)}")
                flag = False

        if flag:
            return val
        
    return None


coordinates1 = {
    "ap": -1.6,
    "ml": -.4,
    "dv": 3.3
}

coords2 = {
    "ap": -3.3,
    "ml": .43,
    "dv": 4.4
}

print(find_which_injection('709393', coordinates1))
print(find_which_injection('709393', coords2))

1
2


In [7]:
def find_which_ro(subj_id):
    subj_row = materials_sheet.loc[materials_sheet["Mouse ID"] == int(subj_id)]

    for val in range(1,4):
        if subj_row[f"Virus{val} Injection Type"].values[0] == "RO":
            return val

    return None

print(find_which_ro('709393'))

3


In [8]:
files = glob.glob("./original_spec_files/*.json")

for file in files:
    with open(file) as f:
        data = json.load(f)
        print(data)
        original_procedure = Procedures.model_construct(**data)

    print(original_procedure)

    subj = original_procedure.subject_id

    logging.info(f"Processing subject {subj}")

    for surgery in original_procedure.subject_procedures:
        print(surgery)
        if "protocol_id" not in surgery.keys():
            surgery["protocol_id"] = "dx.doi.org/10.17504/protocols.io.kqdg392o7g25/v1"
            logging.info(f"adding surgery protocol id for subject {subj}")
        elif surgery["protocol_id"] == "unknown":
            logging.info(f"replacing surgery protocol id for subject {subj}")
            surgery["protocol_id"] = "dx.doi.org/10.17504/protocols.io.kqdg392o7g25/v1"
        for subj_procedure in surgery["procedures"]:
            logging.info(f"checking procedure {subj_procedure} for subject {subj}")
            if subj_procedure["procedure_type"] == "Perfusion":
                if "protocol_id" not in subj_procedure.keys():
                    logging.info(f"adding perfusion protocol id for subject {subj}")
                    subj_procedure["protocol_id"] = "dx.doi.org/10.17504/protocols.io.bg5vjy66"
                    
                elif subj_procedure["protocol_id"] == "unknown":
                    logging.info(f"replacing perfusion protocol id for subject {subj}")
                    subj_procedure["protocol_id"] = "dx.doi.org/10.17504/protocols.io.bg5vjy66"

            if subj_procedure["procedure_type"] == "Retro-orbital injection":
                logging.info(f"checking retro-orbital injection for subject {subj}")

                which = find_which_ro(subj)
                if not which:
                    continue
                material = get_inj_material(subj, which)

                logging.info(f"RO material for subject {subj}, inj {which}: {material}")

                subj_procedure["injection_materials"] = [material]

            if subj_procedure["procedure_type"] == "Nanoject injection":
                coords = {}
                coords['ap'] = Decimal(subj_procedure["injection_coordinate_ap"])
                coords['ml'] = Decimal(subj_procedure["injection_coordinate_ml"])
                coords['dv'] = Decimal(subj_procedure["injection_coordinate_depth"][0])
                logging.info(f"coords: {coords}")
                which = find_which_injection(subj, coords)
                if not which:
                    logging.error(f"Could not find which injection for {subj} nanoject injection {coords}")
                    continue
                material = get_inj_material(subj, which)
                logging.info(f"Nanoject material for subject {subj}, inj {which}: {material}")
                subj_procedure["injection_materials"] = [material]
                
                

    original_procedure.write_standard_file(
        output_directory=Path("original_plus_materials"),
        prefix=subj
    )

    
print(sanity_checks)


    # titer = dose / volume, with volume in ml (gc/ml) (translate to ml)

    # perhaps put vehicle in notes field of surgery?

    # for value in [1,2,3]:


{'describedBy': 'https://raw.githubusercontent.com/AllenNeuralDynamics/aind-data-schema/main/src/aind_data_schema/core/procedures.py', 'schema_version': '0.13.3', 'subject_id': '576404', 'subject_procedures': [{'procedure_type': 'Surgery', 'start_date': '2021-07-12', 'experimenter_full_name': '28908', 'iacuc_protocol': '1806', 'animal_weight_prior': None, 'animal_weight_post': None, 'weight_unit': 'gram', 'anaesthesia': None, 'workstation_id': None, 'procedures': [{'procedure_type': 'Perfusion', 'output_specimen_ids': ['576404'], 'protocol_id': 'dx.doi.org/10.17504/protocols.io.bg5vjy66'}], 'notes': None, 'protocol_id': 'dx.doi.org/10.17504/protocols.io.kqdg392o7g25/v1'}], 'specimen_procedures': [], 'notes': None}
describedBy='https://raw.githubusercontent.com/AllenNeuralDynamics/aind-data-schema/main/src/aind_data_schema/core/procedures.py' schema_version='0.13.3' subject_id='576404' subject_procedures=[{'procedure_type': 'Surgery', 'start_date': '2021-07-12', 'experimenter_full_name'

c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_json(
c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_json(
c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, def

{'procedure_type': 'Surgery', 'start_date': '2022-12-06', 'experimenter_full_name': '30333', 'iacuc_protocol': '2109', 'animal_weight_prior': None, 'animal_weight_post': None, 'weight_unit': 'gram', 'anaesthesia': None, 'workstation_id': None, 'procedures': [{'procedure_type': 'Perfusion', 'output_specimen_ids': ['652781'], 'protocol_id': 'dx.doi.org/10.17504/protocols.io.bg5vjy66'}], 'notes': None}
{'describedBy': 'https://raw.githubusercontent.com/AllenNeuralDynamics/aind-data-schema/main/src/aind_data_schema/core/procedures.py', 'schema_version': '0.13.3', 'subject_id': '653153', 'subject_procedures': [{'procedure_type': 'Surgery', 'start_date': '2022-11-03', 'experimenter_full_name': '13040', 'iacuc_protocol': '2109', 'animal_weight_prior': None, 'animal_weight_post': None, 'weight_unit': 'gram', 'anaesthesia': None, 'workstation_id': None, 'procedures': [{'recovery_time': None, 'recovery_time_unit': 'minute', 'injection_duration': None, 'injection_duration_unit': 'minute', 'instru

c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_json(
c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_json(
c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-pac

{'describedBy': 'https://raw.githubusercontent.com/AllenNeuralDynamics/aind-data-schema/main/src/aind_data_schema/core/procedures.py', 'schema_version': '0.13.3', 'subject_id': '667996', 'subject_procedures': [{'procedure_type': 'Surgery', 'start_date': '2023-03-08', 'experimenter_full_name': '13040', 'iacuc_protocol': '2109', 'animal_weight_prior': None, 'animal_weight_post': None, 'weight_unit': 'gram', 'anaesthesia': None, 'workstation_id': None, 'procedures': [{'recovery_time': None, 'recovery_time_unit': 'minute', 'injection_duration': None, 'injection_duration_unit': 'minute', 'instrument_id': None, 'procedure_type': 'Retro-orbital injection', 'injection_volume': None, 'injection_volume_unit': 'microliter', 'injection_eye': None}], 'notes': None}, {'procedure_type': 'Surgery', 'start_date': '2023-04-12', 'experimenter_full_name': '13040', 'iacuc_protocol': '2109', 'animal_weight_prior': None, 'animal_weight_post': None, 'weight_unit': 'gram', 'anaesthesia': None, 'workstation_id'

c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_json(
c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_json(
c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-pac

{'procedure_type': 'Surgery', 'start_date': '2023-12-01', 'experimenter_full_name': '30509', 'iacuc_protocol': '2109', 'animal_weight_prior': None, 'animal_weight_post': None, 'weight_unit': 'gram', 'anaesthesia': None, 'workstation_id': None, 'procedures': [{'procedure_type': 'Perfusion', 'output_specimen_ids': ['708368'], 'protocol_id': 'dx.doi.org/10.17504/protocols.io.bg5vjy66'}], 'notes': None}
{'describedBy': 'https://raw.githubusercontent.com/AllenNeuralDynamics/aind-data-schema/main/src/aind_data_schema/core/procedures.py', 'schema_version': '0.13.3', 'subject_id': '708369', 'subject_procedures': [{'procedure_type': 'Surgery', 'start_date': '2023-10-27', 'experimenter_full_name': '30509', 'iacuc_protocol': '2109', 'animal_weight_prior': None, 'animal_weight_post': None, 'weight_unit': 'gram', 'anaesthesia': None, 'workstation_id': None, 'procedures': [{'recovery_time': None, 'recovery_time_unit': 'minute', 'injection_duration': None, 'injection_duration_unit': 'minute', 'instru

c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_json(
c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_json(
c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-pac

['Template', nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, 'Chat-IRES-Cre;Slc6a3-T2A-FlpO(ND)', 670784, 670784, 676411, 676410, 676407, 676403, 691966, 691965, 694593, 694590, 694589, 694588, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, 'ePet-Cre', 650008, 650009, 650010, 650011, 667241, 667242, 667244, 669528, 669530, 669533, 673887, 673893, 673894, 673896, 677995, 677993, 677992, 697837, 697836, nan, 'Slc6a4-Cre', 652779, 652781, 669973, 669974, 669977, 673286, 673288, 673289, 684436, 684434, 685592, 685590, 685586, 704473, 704471, 704470, 704469, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, 'xx', nan, 'Sert-FlpO', 678601, 678600, 678599, 678991, 678988, 678987, 686488, 686485, 686486, 686487, 686484, 686483, 703261, 703260, 703259, 703258, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan

c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_json(
c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_json(
c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-pac